In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import jax_cfd.base as cfd
import jax_cfd.base.grids as grids
import jax_cfd.spectral as spectral

from jax_cfd.spectral import equations
from jax_cfd.spectral import time_stepping

The first equation that will be tested is the 2D Euler Equation. The Euler equation related to the Navier-Stokes Equation where the visocity coefficient is 0. Hence the Euler equation is 

$$
\frac{\partial \mathbf{u}}{\partial t}
+(\mathbf{u}\cdot\nabla)\mathbf{u}
=-\nabla p
$$
with
$$\nabla \cdot u = 0 $$
The numerical method to be used is the runge kutta method with order 4. We will use the Taylor-Green inital condition where $u_1 = sin(x)cos(y)$ and $u_2 = -cos(x)sin(y)$

In [3]:
N = 128 #the number of point along the x and y axis
Length = 2 * jnp.pi #The domain will be on [0,2pi] x [0, 2pi]
max_velocity = 7
grid = grids.Grid((N,N), domain = ((0, Length), (0, Length)))
dt = cfd.equations.stable_time_step(max_velocity, .5, 0, grid)
x, y = grid.axes()
X, Y = jnp.meshgrid(x, y, indexing="ij")

equation = equations.NavierStokes2D(viscosity=0.0, grid=grid, drag=0.0, smooth=True, forcing_fn=None) #the euler equation with viscosity = 0.0

#Use runge_kutte to calculate
step_fn = time_stepping.crank_nicolson_rk4(equation, dt)

#Use for the same condition
final_time = 25.0
outer_steps = 10
inner_steps = (final_time // dt) // 10

trajectory_fn = cfd.funcutils.trajectory(
    cfd.funcutils.repeated(step_fn, inner_steps), outer_steps)

#initial condition (Using Taylor Green Condition)
omega0 = 2 * jnp.sin(X) * jnp.sin(Y) 
vorticity_hat0 = jnp.fft.rfftn(omega0)

_, trajectory = trajectory_fn(vorticity_hat0) #compute the trajectory of the Euler Equation with Taylor-Green Condition

In the 2D incompressible flow, the voriticty is defined as

$$\omega = \frac{\partial u_2}{\partial x} - \frac{\partial u_1}{\partial y}$$

so for this example, it is $\omega(x,y, 0) = 2sin(x)cos(x)$ where $u_1 = sin(x)cos(y)$ and $u_2 = -cos(x)sin(y)$

In [5]:
print(type(trajectory))
print(trajectory.shape)
print(trajectory.dtype)

<class 'jaxlib._jax.ArrayImpl'>
(10, 128, 65)
complex64


In [6]:
vorticity_trajectory = jnp.fft.irfftn(trajectory,s=(N, N),axes=(-2, -1)) #convert trajectory back from fourier transform

In [7]:
# Download using pandas
vorticity_np = np.array(vorticity_trajectory)

filename = r"C:\Users\Hoang\Downloads\Euler_vorticity.xlsx"

with pd.ExcelWriter(filename, engine="openpyxl") as writer:

    for i in range(vorticity_np.shape[0]): #for each for the column

        df = pd.DataFrame(vorticity_np[i],index=np.array(x),columns=np.array(y)) #same the columns as dataframe

        df.to_excel(writer,sheet_name=f"t_{i:03d}") #use 3 decimal point

print(f"Saved: {filename}")

Saved: C:\Users\Hoang\Downloads\Euler_vorticity.xlsx


Now we will tested is the 2D incompressible Navier-Stoke Equation. The Taylor-Green initial condition will be used again.

$$
\frac{\partial \mathbf{u}}{\partial t}
+(\mathbf{u}\cdot\nabla)\mathbf{u}
=-\nabla p + \nu^2\mathbf{u}
$$

with 

$$\nabla \cdot u = 0 $$

In [9]:
N = 128 #the number of point along the x and y axis
Length = 2 * jnp.pi #The domain will be on [0,2pi] x [0, 2pi]
max_velocity = 7
viscosity = 1.0 #let the viscosity to be one
grid = grids.Grid((N,N), domain = ((0, Length), (0, Length)))
dt = cfd.equations.stable_time_step(max_velocity, viscosity, 0, grid)
x, y = grid.axes()
X, Y = jnp.meshgrid(x, y, indexing="ij")

equation = equations.NavierStokes2D(viscosity, grid=grid, drag=0.0, smooth=True, forcing_fn=None) #the incompressible navier-stoke with viscosity coefficeint of 1

#Use runge_kutte to calculate
step_fn = time_stepping.crank_nicolson_rk4(equation, dt)

#Use for the same condition
final_time = 25.0
outer_steps = 10
inner_steps = (final_time // dt) // 10

trajectory_fn = cfd.funcutils.trajectory(
    cfd.funcutils.repeated(step_fn, inner_steps), outer_steps)

#initial condition (Using Taylor Green Condition)
omega0 = 2 * jnp.sin(X) * jnp.sin(Y) 
vorticity_hat0 = jnp.fft.rfftn(omega0)

_, trajectory = trajectory_fn(vorticity_hat0) #compute the trajectory of the Euler Equation with Taylor-Green Condition

In [ ]:
vorticity_trajectory = jnp.fft.irfftn(trajectory,s=(N, N),axes=(-2, -1)) #convert trajectory back from fourier transform

# Download using pandas
vorticity_np = np.array(vorticity_trajectory)

filename = r"C:\Users\Hoang\Downloads\Navier_Stoke_vorticity.xlsx"

with pd.ExcelWriter(filename, engine="openpyxl") as writer:

    for i in range(vorticity_np.shape[0]): #for each for the column

        df = pd.DataFrame(vorticity_np[i],index=np.array(x),columns=np.array(y)) #same the columns as dataframe

        df.to_excel(writer,sheet_name=f"t_{i:03d}") #use 3 decimal point

print(f"Saved: {filename}")